# Clasification Labeling

The Goal of this Notebook is to find a way to label a training dataset for classification purposes

In [50]:
import pandas as pd

In [51]:
df = pd.read_csv("data/GBPUSD/minutes.csv")

In [52]:
df.head()

,time,open,high,low,close,tick_volume,spread,real_volume
0,915433980,1.6686,1.6686,1.6686,1.6686,1,50,0
1,915434040,1.6682,1.6685,1.6682,1.6685,4,50,0
2,915434100,1.6689,1.6689,1.6689,1.6689,1,50,0
3,915434160,1.6687,1.6687,1.6685,1.6685,3,50,0
4,915434220,1.6691,1.6691,1.6691,1.6691,1,50,0


In [53]:
def normalize_by_window(df, window_size=120, chunk_size=10):
    col_numbers = range(1, window_size+1)
    col_split = [col_numbers[i:i + chunk_size] for i in range(0, len(col_numbers), chunk_size)]
    df['window_min'] = df['low']
    df['window_max'] = df['high']

    for col_range in col_split:
        high_cols = []
        low_cols = []
        
        for x in col_range:
            high_col_name = f"high-{x}"
            df[high_col_name] = df['high'].shift(x)
            high_cols.append(high_col_name)
        high_cols_inclusive = high_cols + ['window_max']
        df['window_max'] = df[high_cols_inclusive].max(axis=1)
        df.drop(columns=high_cols, inplace=True)

        for x in col_range:
            low_col_name = f"low-{x}"
            df[low_col_name] = df['low'].shift(x)
            low_cols.append(low_col_name)
        low_cols_inclusive = low_cols + ['window_min']
        df['window_min'] = df[low_cols_inclusive].min(axis=1)
        df.drop(columns=low_cols, inplace=True)


    df['open_normalized'] = (df['open'] - df['window_min'])/(df['window_max'] - df['window_min'])
    df['high_normalized'] = (df['high'] - df['window_min'])/(df['window_max'] - df['window_min'])
    df['low_normalized'] = (df['low'] - df['window_min'])/(df['window_max'] - df['window_min'])
    df['close_normalized'] = (df['close'] - df['window_min'])/(df['window_max'] - df['window_min'])

In [54]:
WINDOW_SIZE = 1200
normalize_by_window(df, window_size=WINDOW_SIZE, chunk_size=20)
df = df[WINDOW_SIZE:]

In [55]:
(df.close_normalized - df.open_normalized).abs().mean()

0.01511572796869776

In [66]:
def label_df(df, window_size=20, multiplier=3):
    df['candle'] = df['close_normalized'] - df['open_normalized']
    mean_candle = df['candle'].abs().mean()
    future_cols = []
    sum_cols = []
    sum_cum = 1
    for x in range(-1, -window_size-1, -1):
        col_name = f'candle-{abs(x)}'
        sum_name = f'sum-{sum_cum}'
        df[col_name] = df['candle'].shift(x)
        future_cols.append(col_name)
        df[sum_name] = df[future_cols].sum(axis=1)
        sum_cols.append(sum_name)
        sum_cum += 1

    df['target'] = 0

    df['prev_candle'] = df['candle'].shift(1)
    df['prev_close'] = df['close'].shift(1)
    df['prev_open'] = df['open'].shift(1)

    mask = (df[sum_cols].max(axis=1) > mean_candle*multiplier) & (df['close_normalized']>0.4) & ((df['candle']>mean_candle*2) | (df['prev_candle'] > 0) | (df['close'] > df[['prev_close', 'prev_open']].max(axis=1))) & (df['candle'] > 0)

    df.loc[mask, 'target'] = 1

    mask_down = (df[sum_cols].min(axis=1) < -mean_candle*multiplier) & (df['close_normalized']<0.6) & ((df['candle']<-mean_candle*2) | (df['prev_candle'] < 0) | (df['close'] < df[['prev_close', 'prev_open']].min(axis=1))) & (df['candle'] < 0)

    df.loc[mask_down, 'target'] = 2
    df.drop(columns=future_cols, inplace=True)
    df.drop(columns=sum_cols, inplace=True)

In [67]:
label_df(df)
df.target.value_counts()

target
0    7386255
2     814692
1     780547
Name: count, dtype: int64

In [58]:
drop_cols = ['tick_volume', 'spread', 'real_volume', 'window_min', 'window_max', 'prev_candle', 'prev_close', 'prev_open']
df.drop(columns=drop_cols, inplace=True)

In [59]:
df['time'] = pd.to_datetime(df['time'], unit='s')

In [60]:
df[df['target']==1].tail(20)

,time,open,high,low,close,open_normalized,high_normalized,low_normalized,close_normalized,candle,target
8982565,2024-04-08 10:58:00,1.26243,1.26254,1.26236,1.26253,0.720567,0.736170,0.710638,0.734752,0.014184,1
8982566,2024-04-08 10:59:00,1.26254,1.26262,1.26250,1.26260,0.736170,0.747518,0.730496,0.744681,0.008511,1
8982567,2024-04-08 11:00:00,1.26261,1.26275,1.26257,1.26264,0.746099,0.765957,0.740426,0.750355,0.004255,1
8982568,2024-04-08 11:01:00,1.26265,1.26275,1.26258,1.26266,0.751773,0.765957,0.741844,0.753191,0.001418,1
8982569,2024-04-08 11:02:00,1.26264,1.26277,1.26262,1.26275,0.750355,0.768794,0.747518,0.765957,0.015603,1
8982573,2024-04-08 11:06:00,1.26258,1.26280,1.26250,1.26279,0.741844,0.773050,0.730496,0.771631,0.029787,1
8982637,2024-04-08 12:10:00,1.26175,1.26182,1.26169,1.26177,0.666667,0.677273,0.657576,0.669697,0.003030,1
8982638,2024-04-08 12:11:00,1.26175,1.26179,1.26174,1.26179,0.666667,0.672727,0.665152,0.672727,0.006061,1
8982643,2024-04-08 12:16:00,1.26151,1.26174,1.26146,1.26174,0.630303,0.665152,0.622727,0.665152,0.034848,1
8982644,2024-04-08 12:17:00,1.26173,1.26180,1.26170,1.26177,0.663636,0.674242,0.659091,0.669697,0.006061,1


In [61]:
df['candle'].abs().std()

0.018923739026876073

In [62]:
df['candle'].abs().mean()

0.01511572796869776

In [63]:
df.loc[2808299]

time                2007-05-04 10:26:00
open                             1.9861
high                             1.9861
low                               1.986
close                            1.9861
open_normalized                0.182692
high_normalized                0.182692
low_normalized                 0.173077
close_normalized               0.182692
candle                              0.0
target                                0
Name: 2808299, dtype: object

In [64]:
df['candle'].abs().mean()

0.01511572796869776

In [65]:
df.loc[2808302, 'candle']

0.0